In [79]:
from typing import List, Dict,  Set, Any, Optional
from upath import UPath
from dataclasses import dataclass

import polars as pl
from trinary import Unknown, weakly, strictly

UNKNOWN = "trinary.Unknown"


In [80]:
@dataclass
class context:
    rule_name:  Optional[str]
    DIM_1:      Optional[str]
    DIM_2:      Optional[str]
    DIM_3:      Optional[str]

CONTEXT = context(rule_name="rule_1", DIM_1="A", DIM_2="1", DIM_3="trinary.Unknown")


rules = pl.DataFrame({  "rule_name": ["rule_1", "rule_2", "rule_3"],
                        "DIM_1": ["A", "B", "C"],
                        "DIM_2": ["1", "2", "3"],
                        "DIM_3": ["X", "trinary.Unknown", "trinary.Unknown"]
                    })

dimensions = ["DIM_1", "DIM_2", "DIM_3"]


rules

rule_name,DIM_1,DIM_2,DIM_3
str,str,str,str
"""rule_1""","""A""","""1""","""X"""
"""rule_2""","""B""","""2""","""trinary.Unknown"""
"""rule_3""","""C""","""3""","""trinary.Unknown"""


In [52]:


def apply_context_rules_engine(CONTEXT: object, 
                               rules: pl.DataFrame, 
                               dimensions: Dict[str, Any]) -> pl.DataFrame:
    # Validate Rules
    if rules is None or rules.is_empty():
        raise ValueError("No rules specified.")

    # Validate Dimensions
    invalid_dimensions = [dim not in rules.columns for dim in dimensions]
    if any(invalid_dimensions):
        raise ValueError("Invalid dimensions specified.")

    # Initialize rule columns
    rules = rules.with_columns([
        pl.lit(0).alias("rule_softmatch_count"),
        pl.lit(0).alias("context_softmatch_count"),
        pl.lit(0).alias("hard_match_count"),
        pl.lit(None).alias("dropped"),
        pl.lit(None).alias("dropped_by_value"),
        pl.lit(None).alias("dropped_by_dimension"),
        pl.lit(None).alias("drop_order")
    ])

    drop_order = 0

    # Loop through dimensions and apply the engine.
    for dimension in dimensions:

        drop_order += 1

        if getattr(CONTEXT, dimension) is not None and dimension in rules.columns:


            context_value = getattr(CONTEXT, dimension)
            print(f"context_value: {context_value}")
            
            dimension_values_raw = rules[dimension].to_list()
            dimension_values = [x if x != "trinary.Unknown" else Unknown for x in dimension_values_raw]
            print(f"dimension_values: {dimension_values}")

            # Comparison 1: Left Side Soft Matches - where the rule does not specify a value
            filter1 = [x is Unknown for x in dimension_values]
            print(f"filter1: {filter1}")

            # Comparison 2: Right Side Soft Matches - Where the Context does not specify a value
            filter2 = [context_value is Unknown for x in dimension_values]
            print(f"filter2: {filter2}")

            # Comparison 3: Hard matches between the ruleset and context
            filter3 = [strictly(x == context_value) for x in dimension_values]

            print(f"filter3: {filter3}")

            #Convert arrays to series
            series_filter1 = pl.Series("filter1", filter1)
            series_filter2 = pl.Series("filter2", filter2)
            series_filter3 = pl.Series("filter3", filter3)


            rules = rules.with_columns(
                rule_softmatch_count=       rules["rule_softmatch_count"]       + series_filter1.cast(pl.Int8()),
                context_softmatch_count=    rules["context_softmatch_count"]    + series_filter2.cast(pl.Int8()),
                hard_match_count=           rules["hard_match_count"]           + series_filter3.cast(pl.Int8())
            )


            # Apply filters to determine dropped rules
            keepers = series_filter1 | series_filter2 | series_filter3
            dropped_mask = pl.when(rules["dropped"].is_null() & ~keepers)

            rules = rules.with_columns([
                (dropped_mask.then(drop_order).otherwise(pl.col("drop_order"))).alias("drop_order"),
                (dropped_mask.then(dimension).otherwise(pl.col("dropped_by_value"))).alias("dropped_by_value"),
                (dropped_mask.then(pl.lit(f"{dimension}")).otherwise(pl.col("dropped_by_dimension"))).alias("dropped_by_dimension"),
                (dropped_mask.then(True).otherwise(pl.col("dropped"))).alias("dropped")
            ])

    # Indicate which rules still qualify
    rules = rules.with_columns(pl.col("dropped").is_null().alias("keep"))

    return rules


# Optimized code with correct syntax using the 'assign' method


# Update rule counts based on filters using the 'assign' method




In [53]:

apply_context_rules_engine(CONTEXT=CONTEXT, rules=rules, dimensions=dimensions)

context_value: A
dimension_values: ['A', 'B', 'C']
filter1: [False, False, False]
filter2: [False, False, False]
filter3: [True, False, False]
context_value: 1
dimension_values: ['1', '2', '3']
filter1: [False, False, False]
filter2: [False, False, False]
filter3: [True, False, False]
context_value: Unknown
dimension_values: ['X', Unknown, Unknown]
filter1: [False, True, True]
filter2: [True, True, True]
filter3: [False, False, False]


rule_name,DIM_1,DIM_2,DIM_3,rule_softmatch_count,context_softmatch_count,hard_match_count,dropped,dropped_by_value,dropped_by_dimension,drop_order,keep
str,str,str,str,i32,i32,i32,bool,str,str,i32,bool
"""rule_1""","""A""","""1""","""X""",0,1,2,null,null,null,null,true
"""rule_2""","""B""","""2""","""trinary.Unknown""",1,1,0,true,"""B""","""DIM_1""",1,false
"""rule_3""","""C""","""3""","""trinary.Unknown""",1,1,0,true,"""C""","""DIM_1""",1,false


In [45]:
"X" == Unknown

False

In [ ]:
import ibis
import ibis.expr.types as ir

def apply_context_rules_engine_ibis(CONTEXT: object, rules: ir.Table, dimensions: Dict[str, Any]):

    LOCAL_CONTEXT = CONTEXT

    # Validate Dimensions
    valid_dimensions = all(dim in rules.columns for dim in dimensions)
    if not valid_dimensions:
        raise ValueError("Invalid dimensions specified.")

    # Validate Rules
    if rules is None or len(rules) == 0:
        raise ValueError("No rules specified.")

    # Apply Rules
    if rules is not None and len(rules) > 0:
        # Initialize rule columns
        rules = rules.mutate(
            rule_softmatch_count=ibis.literal(0),
            context_softmatch_count=ibis.literal(0),
            hard_match_count=ibis.literal(0),
            dropped=ibis.NA,
            dropped_by=ibis.NA
        )

        drop_order = 0

        for dimension in dimensions:
            drop_order += 1

            if getattr(LOCAL_CONTEXT, dimension) is not None and dimension in rules.columns:

                context_value = getattr(CONTEXT, dimension)
                print(f"context_value: {context_value}")
                
                dimension_values_raw = rules[dimension].to_list()
                dimension_values = [x if x != "trinary.Unknown" else Unknown for x in dimension_values_raw]
                print(f"dimension_values: {dimension_values}")

                # Comparison 1: Left Side Soft Matches - where the rule does not specify a value
                filter1 = [x is Unknown for x in dimension_values]
                print(f"filter1: {filter1}")

                # Comparison 2: Right Side Soft Matches - Where the Context does not specify a value
                filter2 = [context_value is Unknown for x in dimension_values]
                print(f"filter2: {filter2}")

                # Comparison 3: Hard matches between the ruleset and context
                filter3 = [strictly(x == context_value) for x in dimension_values]

                print(f"filter3: {filter3}")

                #Convert arrays to ibis colums
                
                series_filter1 = pl.Series("filter1", filter1)
                series_filter2 = pl.Series("filter2", filter2)
                series_filter3 = pl.Series("filter3", filter3)

                rules = rules.mutate(
                    rule_softmatch_count=       rules.rule_softmatch_count + series_filter1.cast('int8'),
                    context_softmatch_count=    rules.context_softmatch_count + series_filter2.cast('int8'),
                    hard_match_count=           rules.hard_match_count + series_filter3.cast('int8')
                )

                droppers = filter1 | filter2 | filter3
                rules = rules.mutate(
                    drop_order=ibis.where(rules.dropped.isnull() & droppers, drop_order, rules.drop_order),
                    dropped_by=ibis.where(rules.dropped.isnull() & droppers, dimension, rules.dropped_by),
                    dropped=ibis.where(rules.dropped.isnull() & droppers, True, rules.dropped)
                )

    rules = rules.mutate(keep=rules.dropped.isnull())

    return rules


In [66]:
import ibis
import ibis.expr.types as ir
from typing import Dict, Any
from ibis import _

UNKNOWN = "trinary.Unknown"

def apply_context_rules_engine_ibis(CONTEXT: object, rules: ir.Table, dimensions: Dict[str, Any]) -> ir.Table:

    LOCAL_CONTEXT = CONTEXT

    # Validate Dimensions
    valid_dimensions = all(dim in rules.columns for dim in dimensions)
    if not valid_dimensions:
        raise ValueError("Invalid dimensions specified.")

    # Validate Rules
    if rules.count().execute() == 0:
        raise ValueError("No rules specified.")

    # Apply Rules
    rules = rules.mutate(
        rule_softmatch_count=ibis.literal(0),
        context_softmatch_count=ibis.literal(0),
        hard_match_count=ibis.literal(0),
        dropped=ibis.NA,
        dropped_by=ibis.NA,
        filter_all_false = ibis.literal(False),
        filter_all_true = ibis.literal(True)


    )

    drop_order = 0

    for drop_order, dimension in enumerate(dimensions, start=1):
        if getattr(LOCAL_CONTEXT, dimension) is not None and dimension in rules.columns:


            filter1 = rules[dimension] == UNKNOWN

            filter1 = rules.mutate(
                filter_1=ibis.case()
                    .when(ibis._[dimension]  == UNKNOWN , ibis._.filter_all_true)
                    .else_( ibis._.filter_all_false)
                    .end(),
                dtype=rules[dimension].type()
            ).get_column('filter1')


            if getattr(LOCAL_CONTEXT, dimension) == UNKNOWN:
                filter2 = rules.mutate(
                    filter2= ibis._.filter_all_true
                ).get_column('filter2')
            else:
                filter2 = rules.mutate(
                    filter2= ibis._.filter_all_false
                ).get_column('filter2')

            context_value = getattr(LOCAL_CONTEXT, dimension)
            filter3 = rules.mutate( ibis._[dimension] == ibis.literal(context_value)).get_column('filter3')

            filter1 = filter1.fillna(False)
            filter2 = filter2.fillna(False)
            filter3 = filter3.fillna(False)

            rules = rules.mutate(
                rule_softmatch_count=rules.rule_softmatch_count + filter1.cast('int8'),
                context_softmatch_count=rules.context_softmatch_count + filter2.cast('int8'),
                hard_match_count=rules.hard_match_count + filter3.cast('int8')
            )

            droppers = filter1 | filter2 | filter3
            rules = rules.mutate(
                drop_order=ibis.where(rules.dropped.isnull() & droppers, drop_order, rules.drop_order),
                dropped_by=ibis.where(rules.dropped.isnull() & droppers, dimension, rules.dropped_by),
                dropped=ibis.where(rules.dropped.isnull() & droppers, True, rules.dropped)
            )

    rules = rules.mutate(keep=rules.dropped.isnull())

    return rules


In [69]:

rules_ibis = ibis.set_backend('polars')
rules_ibis = ibis.memtable(rules)

apply_context_rules_engine_ibis(CONTEXT=CONTEXT, rules=rules_ibis, dimensions=dimensions)

/var/folders/nl/6hxx812s53s7g6gws32khz2m0000gn/T/ipykernel_88081/776145991.py:42: FutureWarning: `where` is deprecated as of v7.0; use `ibis.ifelse` instead
  dropped_by=ibis.where((rules['dropped'].isnull()) & (filter1 | filter2 | filter3), ibis.literal(dimension), rules['dropped_by']),
/var/folders/nl/6hxx812s53s7g6gws32khz2m0000gn/T/ipykernel_88081/776145991.py:43: FutureWarning: `where` is deprecated as of v7.0; use `ibis.ifelse` instead
  dropped=ibis.where((rules['dropped'].isnull()) & (filter1 | filter2 | filter3), ibis.literal(True), rules['dropped'])


InputTypeError: Unable to infer datatype of value Unknown with type <class 'trinary.UnknownClass'>

In [84]:
import ibis
import ibis.expr.types as ir
from typing import Dict, Any



def apply_context_rules_engine_ibis(CONTEXT: object, rules: ir.Table, dimensions: List[Any], keep_all: bool=False) -> ir.Table:
    
    # Validate Dimensions
    valid_dimensions = all(dim in rules.columns for dim in dimensions)
    if not valid_dimensions:
        raise ValueError("Invalid dimensions specified.")
    
    # Validate Rules
    if rules.count().execute() == 0:
        raise ValueError("No rules specified.")
    
    # Initialization
    rules = rules.mutate(
        rule_softmatch_count=ibis.literal(0),
        context_softmatch_count=ibis.literal(0),
        hard_match_count=ibis.literal(0),
        dropped=ibis.NA,
        dropped_by=ibis.NA,
        filter_all_false=ibis.literal(False),
        filter_all_true=ibis.literal(True)
    )
    
    # Apply Rules
    for dimension in dimensions:
        if getattr(CONTEXT, dimension) is not None and dimension in rules.columns:

            context_value = getattr(CONTEXT, dimension)
            
            filter1 = (rules[dimension] == UNKNOWN).ifelse(ibis.literal(True), ibis.literal(False)).name('filter1')
            filter2 = ibis.literal(UNKNOWN == context_value).ifelse(ibis.literal(True), ibis.literal(False)).name('filter2')
            filter3 = (rules[dimension] == ibis.literal(context_value)).name('filter3')
            
            rules = rules.mutate(
                rule_softmatch_count=   rules['rule_softmatch_count']       + filter1.cast('int8'),
                context_softmatch_count=rules['context_softmatch_count']    + filter2.cast('int8'),
                hard_match_count=       rules['hard_match_count']           + filter3.cast('int8'),

                dropped_by=             ibis.ifelse( (rules['dropped'].isnull()) & ~(filter1 | filter2 | filter3), 
                                                        ibis.literal(dimension), 
                                                        rules['dropped_by']),
                dropped=                ibis.ifelse( (rules['dropped'].isnull()) & ~(filter1 | filter2 | filter3), 
                                                        ibis.literal(True), 
                                                        rules['dropped'])
            )
    
    rules = rules.mutate(keep=rules['dropped'].isnull())
    

    if keep_all:


    
    return rules


In [88]:
rules_ibis = ibis.set_backend('polars')
rules_ibis = ibis.memtable(rules)

rules_applied = apply_context_rules_engine_ibis(CONTEXT=CONTEXT, rules=rules_ibis, dimensions=dimensions)

rules_applied.filter(_.keep == True)#.execute()

r0 := InMemoryTable
  data:
    PolarsDataFrameProxy:
      shape: (3, 4)
      ┌───────────┬───────┬───────┬─────────────────┐
      │ rule_name ┆ DIM_1 ┆ DIM_2 ┆ DIM_3           │
      │ ---       ┆ ---   ┆ ---   ┆ ---             │
      │ str       ┆ str   ┆ str   ┆ str             │
      ╞═══════════╪═══════╪═══════╪═════════════════╡
      │ rule_1    ┆ A     ┆ 1     ┆ X               │
      │ rule_2    ┆ B     ┆ 2     ┆ trinary.Unknown │
      │ rule_3    ┆ C     ┆ 3     ┆ trinary.Unknown │
      └───────────┴───────┴───────┴─────────────────┘

r1 := Project[r0]
  rule_name:               r0.rule_name
  DIM_1:                   r0.DIM_1
  DIM_2:                   r0.DIM_2
  DIM_3:                   r0.DIM_3
  rule_softmatch_count:    0
  context_softmatch_count: 0
  hard_match_count:        0
  dropped:                 None
  dropped_by:              None
  filter_all_false:        False
  filter_all_true:         True

r2 := Project[r1]
  rule_name:               r1.rule_name
  DIM_1:                   r1.DIM_1
  DIM_2:                   r1.DIM_2
  DIM_3:                   r1.DIM_3
  rule_softmatch_count:    r1.rule_softmatch_count + Cast(IfElse(bool_expr=r1.DIM_1 == 'trinary.Unknown', true_expr=True, false_null_expr=False), to=int8)
  context_softmatch_count: r1.context_softmatch_count + Cast(IfElse(bool_expr=False, true_expr=True, false_null_expr=False), to=int8)
  hard_match_count:        r1.hard_match_count + Cast(r1.DIM_1 == 'A', to=int8)
  dropped:                 IfElse(bool_expr=IsNull(r1.dropped) & Not(IfElse(bool_expr=r1.DIM_1 == 'trinary.Unknown', true_expr=True, false_null_expr=False) | IfElse(bool_expr=False, true_expr=True, false_null_expr=False) | r1.DIM_1 == 'A'), true_expr=True, false_null_expr=r1.dropped)
  dropped_by:              IfElse(bool_expr=IsNull(r1.dropped) & Not(IfElse(bool_expr=r1.DIM_1 == 'trinary.Unknown', true_expr=True, false_null_expr=False) | IfElse(bool_expr=False, true_expr=True, false_null_expr=False) | r1.DIM_1 == 'A'), true_expr='DIM_1', false_null_expr=r1.dropped_by)
  filter_all_false:        r1.filter_all_false
  filter_all_true:         r1.filter_all_true

r3 := Project[r2]
  rule_name:               r2.rule_name
  DIM_1:                   r2.DIM_1
  DIM_2:                   r2.DIM_2
  DIM_3:                   r2.DIM_3
  rule_softmatch_count:    r2.rule_softmatch_count + Cast(IfElse(bool_expr=r2.DIM_2 == 'trinary.Unknown', true_expr=True, false_null_expr=False), to=int8)
  context_softmatch_count: r2.context_softmatch_count + Cast(IfElse(bool_expr=False, true_expr=True, false_null_expr=False), to=int8)
  hard_match_count:        r2.hard_match_count + Cast(r2.DIM_2 == '1', to=int8)
  dropped:                 IfElse(bool_expr=IsNull(r2.dropped) & Not(IfElse(bool_expr=r2.DIM_2 == 'trinary.Unknown', true_expr=True, false_null_expr=False) | IfElse(bool_expr=False, true_expr=True, false_null_expr=False) | r2.DIM_2 == '1'), true_expr=True, false_null_expr=r2.dropped)
  dropped_by:              IfElse(bool_expr=IsNull(r2.dropped) & Not(IfElse(bool_expr=r2.DIM_2 == 'trinary.Unknown', true_expr=True, false_null_expr=False) | IfElse(bool_expr=False, true_expr=True, false_null_expr=False) | r2.DIM_2 == '1'), true_expr='DIM_2', false_null_expr=r2.dropped_by)
  filter_all_false:        r2.filter_all_false
  filter_all_true:         r2.filter_all_true

r4 := Project[r3]
  rule_name:               r3.rule_name
  DIM_1:                   r3.DIM_1
  DIM_2:                   r3.DIM_2
  DIM_3:                   r3.DIM_3
  rule_softmatch_count:    r3.rule_softmatch_count + Cast(IfElse(bool_expr=r3.DIM_3 == 'trinary.Unknown', true_expr=True, false_null_expr=False), to=int8)
  context_softmatch_count: r3.context_softmatch_count + Cast(IfElse(bool_expr=True, true_expr=True, false_null_expr=False), to=int8)
  hard_match_count:        r3.hard_match_count + Cast(r3.DIM_3 == 'trinary.Unknown', to=int8)
  dropped:                 IfElse(bool_expr=IsNull(r3.dropped) & Not(IfElse(bool_expr=r3.DIM_3 == 

In [2]:
from mountainash_utils_rules import RulesEngine, DimensionsMetadata, Dimension, MatchStrategy
from mountainash_data import IbisDataFrame, DataFrameFactory
from pydantic import BaseModel
from typing import Literal

# Step 1: Define the dimensions
dimensions = DimensionsMetadata(dimensions=[
    Dimension(dimension_name="product_type", match_strategy=MatchStrategy.EXACT, data_type=str),
    Dimension(dimension_name="loan_purpose", match_strategy=MatchStrategy.EXACT, data_type=str),
    Dimension(dimension_name="lvr", match_strategy=MatchStrategy.RANGE, data_type=float, range_min_field="lvr_min", range_max_field="lvr_max"),
    Dimension(dimension_name="total_lending", match_strategy=MatchStrategy.RANGE, data_type=float, range_min_field="total_lending_min", range_max_field="total_lending_max"),
    Dimension(dimension_name="customer_status", match_strategy=MatchStrategy.EXACT, data_type=str),
])

# Step 2: Create the rules data
rules_data = [
    {"rule_id": 1, "product_type": "Fixed", "loan_purpose": "Owner Occupied", "lvr_min": 0, "lvr_max": 80, "total_lending_min": 0, "total_lending_max": 500000, "customer_status": "New", "margin": 0.5},
    {"rule_id": 2, "product_type": "Fixed", "loan_purpose": "Owner Occupied", "lvr_min": 80, "lvr_max": 90, "total_lending_min": 0, "total_lending_max": 500000, "customer_status": "New", "margin": 0.7},
    {"rule_id": 3, "product_type": "Variable", "loan_purpose": "Investment", "lvr_min": 0, "lvr_max": 70, "total_lending_min": 500000, "total_lending_max": 1000000, "customer_status": "Existing", "margin": 0.3},
    # Add more rules as needed
]

rules = DataFrameFactory.create_ibis_dataframe_object_from_dictionary(rules_data, ibis_backend_schema="sqlite")

# Step 3: Set up the RulesEngine
engine = RulesEngine(rules=rules, dimension_metadata=dimensions)

# Step 4: Define the context (customer profile)
class CustomerProfile(BaseModel):
    product_type: Literal["Fixed", "Variable"]
    loan_purpose: Literal["Owner Occupied", "Investment"]
    lvr: float
    total_lending: float
    customer_status: Literal["New", "Existing"]

# Step 5: Apply the rules and get the result
def get_mortgage_margin(profile: CustomerProfile) -> float:
    
    result = engine.apply_context_rules_engine(
        context=profile,
        dimension_names=["product_type", "loan_purpose", "lvr", "total_lending", "customer_status"],
        keep_all=False
    )
   
    
    if result.count() > 0:
        # Return the margin of the highest priority matching rule
        return result.select("margin").materialise()
    else:
        raise ValueError("No matching rules found for the given profile")

# Example usage
customer_profile = CustomerProfile(
    product_type="Fixed",
    loan_purpose="Owner Occupied",
    lvr=75.0,
    total_lending=400000,
    customer_status="New"
)

try:
    margin = get_mortgage_margin(customer_profile)
    print(f"The applicable margin for this customer is: {margin}%")
except ValueError as e:
    print(str(e))

active_dimensions: ['loan_purpose', 'total_lending', 'lvr', 'customer_status', 'product_type']
The applicable margin for this customer is:    margin
0     0.5%


In [14]:

empty_list = []
empty_dict = {}
empty_set = set()
empty_tuple = ()
empty_string = ""

nonempty_list = ["A"]
nonempty_dict1 = {"A"}
nonempty_dict2 = {"A":"A"}
nonempty_set = set("A")
nonempty_tuple = ("A")
nonempty_string = "A"



def is_empty(obj):
    # if isinstance(obj, Mapping):
    #     return len(obj) == 0

    if isinstance(obj, Sequence) and not isinstance(obj, str):
        return len(obj) == 0

    #handle sets
    if isinstance(obj, set):
        return not obj
    


print(f"empty_list: {is_empty(empty_list)}")
print(f"empty_dict: {is_empty(empty_dict)}")
print(f"empty_set: {is_empty(empty_set)}")
print(f"empty_tuple: {is_empty(empty_tuple)}")
print(f"empty_string: {is_empty(empty_string)}")

print(f"nonempty_list: {is_empty(nonempty_list)}")
print(f"nonempty_dict1: {is_empty(nonempty_dict1)}")
print(f"nonempty_dict2: {is_empty(nonempty_dict2)}")
print(f"nonempty_set: {is_empty(nonempty_set)}")
print(f"nonempty_tuple: {is_empty(nonempty_tuple)}")
print(f"nonempty_string: {is_empty(nonempty_string)}")



empty_list: True
empty_dict: None
empty_set: True
empty_tuple: True
empty_string: None
nonempty_list: False
nonempty_dict1: False
nonempty_dict2: None
nonempty_set: False
nonempty_tuple: None
nonempty_string: None


In [19]:
from typing import Mapping, Sequence, Set

def is_empty(obj):
    if isinstance(obj, str):
        return False  # Treat all strings as non-empty
    elif isinstance(obj, (Mapping, Sequence, Set)):
        return len(obj) == 0
    else:
        return False

# Test cases
empty_list = []
empty_dict = {}
empty_set = set()
empty_tuple = ()
empty_object = object()
empty_string = ""

zero = 0
none = None
false = False
true = True
nonempty_list = ["A"]
nonempty_dict1 = {"A"}
nonempty_dict2 = {"A":"A"}
nonempty_set = set("A")
nonempty_tuple = ("A",)
nonempty_string = "A"
nonempty_int = 10


test_cases = [
    ("empty_list", empty_list),
    ("empty_dict", empty_dict),
    ("empty_set", empty_set),
    ("empty_tuple", empty_tuple),
    ("empty_string", empty_string),
    ("empty_object", empty_object),

    ("nonempty_list", nonempty_list),
    ("nonempty_dict1", nonempty_dict1),
    ("nonempty_dict2", nonempty_dict2),
    ("nonempty_set", nonempty_set),
    ("nonempty_tuple", nonempty_tuple),
    ("nonempty_string", nonempty_string),
    ("nonempty_int", nonempty_int),
    ("zero", zero),
    ("none", none),
    ("false", false),
    ("true", true)
]

for name, obj in test_cases:
    print(f"{name}: {is_empty(obj)}")

empty_list: True
empty_dict: True
empty_set: True
empty_tuple: True
empty_string: False
empty_object: False
nonempty_list: False
nonempty_dict1: False
nonempty_dict2: False
nonempty_set: False
nonempty_tuple: False
nonempty_string: False
nonempty_int: False
zero: False
none: False
false: False
true: False
